# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


### We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

H0: mean_HP_Dragon <= mean_HP_Grass

H1: mean_HP_Dragon > mean_HP_Grass

In [3]:
# Filter HP values by primary type
dragon_hp = df.loc[df["Type 1"] == "Dragon", "HP"]
grass_hp = df.loc[df["Type 1"] == "Grass", "HP"]

# Summary statistics
summary = pd.DataFrame({
    "type": ["Dragon", "Grass"],
    "n": [dragon_hp.shape[0], grass_hp.shape[0]],
    "mean_hp": [dragon_hp.mean(), grass_hp.mean()],
    "std_hp": [dragon_hp.std(), grass_hp.std()]
})

display(summary)

# Welch's independent two-sample t-test
alpha = 0.05

t_stat, p_value = st.ttest_ind(
    dragon_hp,
    grass_hp,
    equal_var=False,
    alternative="greater"
)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print("Reject H0: Dragon Pokémon have significantly higher HP than Grass Pokémon.")
else:
    print("Fail to reject H0: There is not enough evidence that Dragon Pokémon have higher HP than Grass Pokémon.")

,type,n,mean_hp,std_hp
0,Dragon,32,83.312500,23.795415
1,Grass,70,67.271429,19.516564


t-statistic: 3.3350
p-value: 0.0008
Reject H0: Dragon Pokémon have significantly higher HP than Grass Pokémon.


At a 5% significance level, we reject the null hypothesis.

The p-value is below 0.05, meaning there is statistically significant evidence that Dragon-type Pokémon have, on average, higher HP than Grass-type Pokémon.

In practical terms, Dragon Pokémon show a higher average HP than Grass Pokémon in this dataset.

### We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.

H0: Legendary and Non-Legendary Pokémon have the same average stat profile.

H1: Legendary and Non-Legendary Pokémon have different average stat profiles.

In [4]:
from statsmodels.multivariate.manova import MANOVA

# Select relevant columns
stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

df_manova = df[stats_cols + ["Legendary"]].dropna().copy()

# Rename columns to avoid problems with spaces/dots in statsmodels formula
df_manova = df_manova.rename(columns={
    "Sp. Atk": "Sp_Atk",
    "Sp. Def": "Sp_Def"
})

# MANOVA test
manova = MANOVA.from_formula(
    "HP + Attack + Defense + Sp_Atk + Sp_Def + Speed ~ Legendary",
    data=df_manova
)

print(manova.mv_test())

                   Multivariate linear model
                                                                
----------------------------------------------------------------
       Intercept         Value  Num DF  Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda  0.0592 6.0000 793.0000 2100.8338 0.0000
         Pillai's trace  0.9408 6.0000 793.0000 2100.8338 0.0000
 Hotelling-Lawley trace 15.8953 6.0000 793.0000 2100.8338 0.0000
    Roy's greatest root 15.8953 6.0000 793.0000 2100.8338 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
          Legendary        Value  Num DF  Den DF  F Value Pr > F
----------------------------------------------------------------
             Wilks' lambda 0.7331 6.0000 793.0000 48.1098 0.0000
            Pillai's trace 0.2669 6.0000 793.

In [5]:
stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
alpha = 0.05

results = []

for stat in stats_cols:
    legendary = df.loc[df["Legendary"] == True, stat]
    non_legendary = df.loc[df["Legendary"] == False, stat]
    
    t_stat, p_value = st.ttest_ind(
        legendary,
        non_legendary,
        equal_var=False,
        alternative="two-sided"
    )
    
    results.append({
        "stat": stat,
        "legendary_mean": legendary.mean(),
        "non_legendary_mean": non_legendary.mean(),
        "t_statistic": t_stat,
        "p_value": p_value,
        "significant_5pct": p_value < alpha
    })

results_df = pd.DataFrame(results)
results_df

,stat,legendary_mean,non_legendary_mean,t_statistic,p_value,significant_5pct
0,HP,92.738462,67.182313,8.981370,1.002691e-13,True
1,Attack,116.676923,75.669388,10.438134,2.520372e-16,True
2,Defense,99.661538,71.559184,7.637078,4.826998e-11,True
3,Sp. Atk,122.184615,68.454422,13.417450,1.551461e-21,True
4,Sp. Def,105.938462,68.892517,10.015697,2.294933e-15,True
5,Speed,100.184615,65.455782,11.475044,1.049016e-18,True


In [6]:
results_df["bonferroni_p_value"] = (results_df["p_value"] * len(stats_cols)).clip(upper=1)
results_df["significant_after_bonferroni"] = results_df["bonferroni_p_value"] < alpha

results_df

,stat,legendary_mean,non_legendary_mean,t_statistic,p_value,significant_5pct,bonferroni_p_value,significant_after_bonferroni
0,HP,92.738462,67.182313,8.981370,1.002691e-13,True,6.016147e-13,True
1,Attack,116.676923,75.669388,10.438134,2.520372e-16,True,1.512223e-15,True
2,Defense,99.661538,71.559184,7.637078,4.826998e-11,True,2.896199e-10,True
3,Sp. Atk,122.184615,68.454422,13.417450,1.551461e-21,True,9.308768e-21,True
4,Sp. Def,105.938462,68.892517,10.015697,2.294933e-15,True,1.376960e-14,True
5,Speed,100.184615,65.455782,11.475044,1.049016e-18,True,6.294098e-18,True


### Legendary vs Non-Legendary Pokémon Stats

Since we are comparing Legendary and Non-Legendary Pokémon across multiple numerical variables at the same time, the most appropriate overall test is a MANOVA.

The null hypothesis states that Legendary and Non-Legendary Pokémon have the same average stat profile across HP, Attack, Defense, Sp. Atk, Sp. Def, and Speed.

At a 5% significance level, the MANOVA test shows a statistically significant difference between the two groups. Therefore, we reject the null hypothesis.

This means that Legendary Pokémon and Non-Legendary Pokémon do not have the same overall stat profile.

To understand where the differences come from, we also performed individual Welch t-tests for each stat. The results show that Legendary Pokémon have higher average values across the main battle stats, and the differences are statistically significant.

In practical terms, Legendary Pokémon are generally stronger than Non-Legendary Pokémon across HP, Attack, Defense, Special Attack, Special Defense, and Speed.

# **Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [7]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


### **We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

H0: mean_price_close <= mean_price_far

H1: mean_price_close > mean_price_far

In [9]:

school = (-118, 34)
hospital = (-122, 37)

def euclidean_distance(lon, lat, point):
    return np.sqrt((lon - point[0])**2 + (lat - point[1])**2)

df["distance_to_school"] = euclidean_distance(df["longitude"], df["latitude"], school)
df["distance_to_hospital"] = euclidean_distance(df["longitude"], df["latitude"], hospital)

df["close_to_school_or_hospital"] = (
    (df["distance_to_school"] < 0.50) |
    (df["distance_to_hospital"] < 0.50)
)

close_prices = df.loc[df["close_to_school_or_hospital"], "median_house_value"]
far_prices = df.loc[~df["close_to_school_or_hospital"], "median_house_value"]

summary = pd.DataFrame({
    "group": ["Close", "Far"],
    "n": [close_prices.shape[0], far_prices.shape[0]],
    "mean_price": [close_prices.mean(), far_prices.mean()],
    "median_price": [close_prices.median(), far_prices.median()],
    "std_price": [close_prices.std(), far_prices.std()]
})

display(summary)

# Welch t-test: close houses are more expensive
alpha = 0.05

t_stat, p_value = st.ttest_ind(
    close_prices,
    far_prices,
    equal_var=False,
    alternative="greater"
)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < alpha:
    print("Reject H0: houses close to a school or hospital are significantly more expensive.")
else:
    print("Fail to reject H0: not enough evidence that houses close to a school or hospital are more expensive.")

,group,n,mean_price,median_price,std_price
0,Close,6829,246951.982135,218200.0,111823.983975
1,Far,10171,180678.441058,149800.0,111019.113084


t-statistic: 37.9923
p-value: 0.000000
Reject H0: houses close to a school or hospital are significantly more expensive.


### School/Hospital Proximity and House Prices

To test whether houses close to either a school or a hospital are more expensive, we first calculated the Euclidean distance from each neighborhood to the school coordinates `(-118, 34)` and the hospital coordinates `(-122, 37)`.

A house was classified as close if its distance to either location was below `0.50`.

Since we are comparing the average `median_house_value` between two independent groups, the appropriate test is a Welch independent two-sample t-test. Welch's test is preferred because it does not assume equal variances between the two groups.

The hypotheses are:

- H0: Houses close to a school or hospital are not more expensive on average.
- H1: Houses close to a school or hospital are more expensive on average.

At a 5% significance level, if the p-value is below 0.05, we reject the null hypothesis.

Based on the test result, we conclude whether there is statistically significant evidence that houses located close to a school or hospital have higher average prices than houses farther away.